# Application of Radial Equilibrium Equation for a Rotor
## Adaptation of the code RE-DES by Lewis (Turbomachines Performance Analysis)

The radial equilibrium equation can be reordened and written as

$$\boxed{
\frac{\text{d}}{\text{d}r} c_x(r)^2 = 2\left( \omega - \frac{c_\theta(r)}{r}\right)
\frac{\text{d} (rc_\theta(r))}{\text{d}r}
}  \tag{1}
$$

that gives the solution
$$
c_x(r) = \sqrt{f(r) + k} \tag{2}
$$
where
$$
f(r) = 2 \int_{r_h}^r \left( \omega - \frac{c_\theta(r)}{r}\right)\text{d} (rc_\theta(r)) \tag{3}
$$

The aim is, given all the data: $Q$, $\omega$, $r_h$, $r_t$ and the function
$c_\theta(r)$, compute $c_x(r)$ and the angle $\beta_2(r)$ that fullfils the
required flowrate.

The necessary modules are imported

In [42]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 2)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

The data for the fan is input in this cell.
The hub to tip ratio and the rms radius, $r_\text{rms} = \sqrt{\frac{r_h^2+r_t^2}{2}}$ are computed, and the swirl distribution is also chosen.
For numerical computations, the $r$ domain is divided in $m$ intervals, and $n$ for output of results. Velocity at $r_{rms}$, $c_{\theta,\text{rms}}=c_\theta(r=r_\text{rms})$, is calculated from Euler's equation, and $c_x$ in $r_\text{rms}$ is as well estimated, assuming that it is the average velocity given by the flow rate, $\overline{c_x}$.

In [43]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 601                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 70                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.4767 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.6837 m/s
cx_rms = 10.3916 m/s


- **Free Vortex**:
  $$ c_\theta = \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
- **Forced Vortex**:
  $$ c_\theta = Ar $$
  where 
  $$ A = \frac{c_{\theta,rms}}{r_{rms}} $$
- **Constant Vortex**:
  $$ c_\theta = c_{\theta,rms} $$
- **Mixed Vortex**:
  $$ c_\theta = A(r-r_{rms}) + \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
  and
  $$ A = \frac{\Delta c_\theta}{r_t - r_h} + \frac{B}{r_t r_h} $$
  where $\Delta c_\theta$ is the variability of $c_\theta$ between the tip and the hub, that is, some kind of "strength" of the vortex
- **Arbitray Vortex**:
  The user can define any function $c_\theta(r)$


In [44]:
# User choice: "free", "forced", "constant", "mixed", "arbitrary"
flow_type = "mixed"

if flow_type == "free":
    B = ctrms*rrms
    ctheta = B/r
    print("Free vortex flow")
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "forced":
    A = ctrms/(rrms)
    ctheta = A*r
    print("Forced vortex flow")
    print("A = {:.4f} 1/s".format(A))
elif flow_type == "constant":
    ctheta = ctrms*np.ones(r.size)
    print("Constant ctheta flow")
    print("ctheta = {:.4f} m/s".format(ctrms))
elif flow_type == "mixed":
    delta_ctheta = 1
    B = ctrms*rrms
    A = delta_ctheta/(rt-rh) + B/(rt*rh)
    ctheta = A*(r-rrms) + B/r
    print("Mixed vortex flow")
    print("A = {:.4f} 1/s".format(A))
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "arbitrary":
    ctheta = 2*omega/3*r-9.08/np.sqrt(r)

Mixed vortex flow
A = 71.9661 1/s
B = 0.2449 m²/s


And now, the function $f(r)$ is computed by numerical integration (Eq. (3))

In [45]:
f = 2.0 * cumulative_trapezoid(omega-np.divide(ctheta,r),
                         np.multiply(r,ctheta),initial=0)

### First approximation

The first aproximation of the value of $k$ is with the assumption that
$c_{x,\text{rms}} = \overline{c_x}$

In [46]:
frms = CubicSpline(r,f)(rrms)
k = cxm*cxm-frms
print("First approximation of k: \n k = {:.4f} m²/s²".format(k))

First approximation of k: 
 k = 49.6024 m²/s²


Values of $DF < 0.6$ and a first estimation of $C_D$ are defined. With this assumptions and data, ${C_L}$, ${σ}$ (solidity) and ${c_x}$ along the entire length of the profile are calculated.

With these data, the chord of the profile along the length of the blade is calculated, thus defining the geometry of the blade.


$$
σ = \frac {cos(β_1) (tan(β_1) - tan(β_2))}{2 D_F - 2  [1 - \frac{cos(β_1)}{cos(β_2) } ] } \tag{4}
$$

$$
C_L = \frac {2  cos(β_m)  (tan(β_1) - tan(β_2))}{σ} - C_Dtan(β_m) \tag{5}
$$

The contribution of Samuel Limonchi (course 2023-24 of MUREM) to this part of the notebook is acknowledged.

In [47]:
DF_target = 0.4
CD = 0.01
Nblades = 5
def solve_fan(k):
    cx = np.sqrt(k + f)
    Q = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    Delta_p0 = rho * omega * r * ctheta
    Delta_p0_avg = 2*trapezoid(np.multiply(r,Delta_p0),r)/(rt*rt-rh*rh)
    data_list = []
    #print("{:^15}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}{:^10}"
    #      .format("Radius","ctheta","cx","alpha2","beta1","beta2","beta_m","Solidity","CD","CL","chord"))
    #print("\u2500"*80)
    for i in range(n):
        rdata = rh + (rt-rh)*i/(n-1)
        cthetadata = float(CubicSpline(r,ctheta)(rdata)) # That is in order to treat it as a float and not an array
        cxans = float(CubicSpline(r,cx)(rdata)) # That is in order to treat it as a float and not an array
        alpha2 = np.rad2deg(np.arctan(cthetadata/cxans))
        beta2 = np.rad2deg(np.arctan((omega*rdata-cthetadata)/cxans))
        beta1 = np.rad2deg(np.arctan(omega*rdata/cxans))
        beta_m = 0.5*(beta1+beta2)
        cosbeta1 = np.cos(np.deg2rad(beta1))
        cosbeta2 = np.cos(np.deg2rad(beta2))
        tanbeta1 = np.tan(np.deg2rad(beta1))
        tanbeta2 = np.tan(np.deg2rad(beta2))
        tanbetam = 0.5*(tanbeta1 + tanbeta2)
        beta_m = np.rad2deg(np.arctan(tanbetam))
        cosbetam = 1 / np.sqrt(1 + tanbetam*tanbetam)
        solidity = (cosbeta1* (tanbeta1 - tanbeta2)) / (2*DF_target - 2 * (1 - (cosbeta1/cosbeta2) ))
        bladespace = 2*np.pi*rdata/Nblades
        chord =  solidity * bladespace
        CL = (2/solidity) * (cosbetam * (tanbeta1 - tanbeta2)) - CD*tanbetam
        Delta_p0_data = rho * omega * rdata * cthetadata
        data_list.append({
            "Radius (m)": rdata,
            "c_theta (m/s)": cthetadata,
            "c_x (m/s)": cxans,
            "alpha_2 (deg)": alpha2,
            "beta_1 (deg)": beta1,
            "beta_2 (deg)": beta2,
            "beta_m (deg)": beta_m,
            "Solidity": solidity,
            "CL": CL,
            "Chord (mm)": chord * 1000,
            "Delta p0 (Pa)": Delta_p0_data
        })
        #print("{:^15.4f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}{:^10.2f}".
        #        format(rdata,cthetadata,cxans,alpha2,beta1,beta2,beta_m,solidity,CD,CL,chord))
    print("Q = {:.3f} m^3/s".format(Q))
    errorQ = np.abs(Qdata-Q)/Qdata * 100
    print("error in flow rate = {:.1e} %".format(errorQ))
    print("Delta p0 avg = {:.2f} Pa".format(Delta_p0_avg))
    errorP0 = np.abs(Delta_p0_target-Delta_p0_avg)/Delta_p0_target * 100
    print("error in pressure rise = {:.1e} %".format(errorP0))
    df = pd.DataFrame(data_list)
    df = df.round({"Chord (mm)":0})
    return df

df = solve_fan(k)

Q = 0.481 m^3/s
error in flow rate = 8.5e-01 %
Delta p0 avg = 76.68 Pa
error in pressure rise = 9.5e+00 %


In [48]:
df

,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Delta p0 (Pa)
0,0.03,3.39,7.04,25.69,47.27,31.02,40.09,0.85,0.86,34.0,31.00
1,0.04,2.26,6.75,18.53,56.19,49.18,52.97,0.37,1.07,20.0,27.41
2,0.05,1.87,6.84,15.32,61.41,57.35,59.50,0.23,1.20,15.0,28.20
3,0.06,1.85,7.37,14.13,63.86,60.75,62.38,0.18,1.25,15.0,33.40
4,0.07,2.05,8.27,13.93,64.68,61.80,63.31,0.17,1.26,16.0,42.99
5,0.08,2.38,9.43,14.18,64.68,61.75,63.29,0.18,1.26,19.0,56.97
6,0.09,2.80,10.76,14.60,64.34,61.22,62.86,0.19,1.24,22.0,75.34
7,0.10,3.29,12.20,15.09,63.86,60.50,62.27,0.20,1.23,26.0,98.11
8,0.11,3.82,13.71,15.58,63.35,59.74,61.65,0.22,1.21,31.0,125.27
9,0.12,4.39,15.26,16.04,62.86,58.99,61.04,0.23,1.19,36.0,156.83


### More precise computation


Instead of estimating $k$ with the assumption of $c_x$ in $r_{rms}$ being the average value, a more accurate computation
can be performed forcing the flow rate to be the input one (equation (5.47) and figure 5.6)

In [49]:
from scipy.optimize import brentq

In [50]:
def QFunction(k):
    cx = np.sqrt(k + f)
    Q_temptative = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    return Qdata-Q_temptative

k = brentq(QFunction,0.5*k,1.5*k)
print("k = {:.4f} m²/s²".format(k))

k = 47.8954 m²/s²


In [51]:
df = solve_fan(k)
df

Q = 0.477 m^3/s
error in flow rate = 3.5e-14 %
Delta p0 avg = 76.68 Pa
error in pressure rise = 9.5e+00 %


,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Delta p0 (Pa)
0,0.03,3.39,6.92,26.09,47.77,31.46,40.59,0.88,0.84,35.0,31.00
1,0.04,2.26,6.63,18.86,56.69,49.72,53.49,0.38,1.07,20.0,27.41
2,0.05,1.87,6.71,15.59,61.85,57.84,59.97,0.23,1.20,15.0,28.20
3,0.06,1.85,7.25,14.35,64.22,61.14,62.76,0.18,1.25,15.0,33.40
4,0.07,2.05,8.16,14.10,64.96,62.10,63.60,0.17,1.26,16.0,42.99
5,0.08,2.38,9.34,14.31,64.90,61.99,63.52,0.18,1.25,19.0,56.97
6,0.09,2.80,10.68,14.71,64.50,61.40,63.03,0.19,1.24,22.0,75.34
7,0.10,3.29,12.13,15.17,63.99,60.65,62.41,0.20,1.23,26.0,98.11
8,0.11,3.82,13.65,15.64,63.46,59.85,61.76,0.22,1.21,31.0,125.27
9,0.12,4.39,15.21,16.10,62.95,59.08,61.13,0.23,1.19,36.0,156.83
